# Pre-training Model Evaluation: Kalite Nasıl Ölçülür?

Bu notebook'ta, birden fazla **base (pre-trained)** modeli yan yana karşılaştırarak, bir modelin ne kadar "iyi" eğitildiğini ölçen temel teknikleri uygulayacağız.

### Kullanım
`MODELS` ve `TOKENIZERS` sözlüklerine istediğin kadar model ekle — tüm testler otomatik olarak hepsinde çalışır.

### Kapsanan Katmanlar

| # | Katman | Section'lar |
|---|--------|-------------|
| 1 | **Loss-Based Metrics** | PPL, BPC, Token Entropy, Sliding Window PPL |
| 2 | **Benchmark (Log-Likelihood)** | Multiple Choice Ranking |
| 3 | **Linguistic Competence** | Subject-Verb Agreement, Word Order, Türkçe Morfoloji |
| 4 | **Factual Knowledge** | Popular vs Rare, Contrastive Margin |
| 5 | **Reasoning & ICL** | Few-shot ICL Curve, Flipped Labels |
| 6 | **Generation Quality & Calibration** | Rep-4, Distinct-2, ECE, Paraphrase Consistency |
| 7 | **Mechanistic Interpretability** | Logit Lens, Attention Map, Top-K Confidence |

---

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import math
import gc

device = "cuda" if torch.cuda.is_available() else "cpu"

def clear_gpu():
    """Her yöntem sonrası GPU belleğini temizle."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def get_device(model):
    """Model device'ini döndür — device_map=auto ile dağılmış modellerde de çalışır."""
    try:
        return next(model.parameters()).device
    except StopIteration:
        return torch.device(device)

print(f"Cihaz: {device}")


In [ ]:
# ============================================================
# Model Yükleme — Base vs Fine-tuned Karşılaştırması
# Aynı taban (Kumru-2B) üzerinde ne kadar değişim olmuş?
# ============================================================

model_configs = {
    #"Kumru-2B (base)": {"id": "vngrs-ai/Kumru-2B", "kwargs": {"torch_dtype": "auto", "device_map": "auto", "attn_implementation": "eager"}},
    #"Kara-Kumru (alican)": {"id": "AlicanKiraz0/Kara-Kumru-v1.0-2B", "kwargs": {"torch_dtype": "auto", "device_map": "auto", "attn_implementation": "eager"}},
    #"gemma4-2b": {"id":"google/gemma-4-E2B","kwargs": {"torch_dtype": "auto", "device_map": "auto", "attn_implementation": "eager"}},
    "gemma4-2b-it": {"id":"google/gemma-4-E2B-it","kwargs": {"torch_dtype": "auto", "device_map": "auto", "attn_implementation": "eager"}}
}

MODELS = {}
TOKENIZERS = {}

for name, cfg in model_configs.items():
    print(f"Yükleniyor: {name} ({cfg['id']})...")
    TOKENIZERS[name] = AutoTokenizer.from_pretrained(cfg["id"])
    m = AutoModelForCausalLM.from_pretrained(cfg["id"], **cfg["kwargs"]);
    MODELS[name] = m.to(device) if "device_map" not in cfg["kwargs"] else m
    clear_gpu()

print(f"\n{len(MODELS)} model yüklendi: {list(MODELS.keys())}")

## Helper Fonksiyonlar

Notebook boyunca tekrar tekrar kullanılan temel yardımcı fonksiyonlar.

In [ ]:
def get_logprob(model, tokenizer, prefix, continuation):
    """Prefix verildiğinde continuation'ın toplam log-probability'si.
    Birçok test (agreement, factual, ICL) bu fonksiyona dayanır."""
    full = prefix + continuation
    inputs = tokenizer(full, return_tensors="pt").to(get_device(model))
    prefix_len = tokenizer(prefix, return_tensors="pt")["input_ids"].shape[1]
    with torch.no_grad():
        logits = model(**inputs).logits[0]
    log_probs = torch.log_softmax(logits, dim=-1)
    ids = inputs["input_ids"][0]
    return sum(log_probs[i-1, ids[i]].item() for i in range(prefix_len, len(ids)))


def greedy_complete(model, tokenizer, prompt, max_tokens=20):
    """Greedy decoding — en yüksek olasılıklı token zinciri."""
    inputs = tokenizer(prompt, return_tensors="pt").to(get_device(model))
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_tokens, do_sample=False)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:],
                            skip_special_tokens=True)


def sampled_generate(model, tokenizer, prompt, max_tokens=200):
    """Sampled generation — diversity ölçümleri için."""
    inputs = tokenizer(prompt, return_tensors="pt").to(get_device(model))
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_tokens,
                             do_sample=True, temperature=0.7, top_p=0.9,
                             repetition_penalty=1.1)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:],
                            skip_special_tokens=True)


print("Helper fonksiyonlar yüklendi: get_logprob, greedy_complete, sampled_generate")

## 1. Perplexity (PPL) ve BPC (Bits Per Character) Hesaplama

### Nedir?
- **Perplexity (PPL):** Modelin bir sonraki token'ı tahmin ederken ortalama kaç seçenek arasında "kararsız" kaldığını gösteren metriktir. Matematiksel olarak cross-entropy loss'un üstel halidir: `PPL = exp(loss)`.
- **BPC (Bits Per Character):** Loss'u karakter başına bit cinsine çevirir. Tokenizer'dan bağımsız bir ölçüm sağlar, bu sayede farklı sözlük boyutlarına sahip modeller adil şekilde karşılaştırılabilir.

$$BPC = \frac{Loss \times \text{Token Sayısı}}{\text{Karakter Sayısı} \times \ln(2)}$$

### Ne bulmayı bekliyoruz?
- İyi eğitilmiş bir model, dili iyi "sıkıştırabilmeli" → **düşük PPL ve BPC**.
- Türkçe ağırlıklı eğitilmiş modellerin Türkçe metinde daha düşük PPL vermesini bekliyoruz.
- Çok büyük PPL (>100) modelin bu dili/domain'i iyi öğrenmediğine işaret eder.

### Nasıl yorumlarız?
| Metrik | İyi | Kötü |
|--------|-----|------|
| PPL | Düşük (ör. 5-30) | Yüksek (ör. 100+) |
| BPC | < 1.5 | > 3.0 |

> **Dikkat:** PPL'i farklı tokenizer'lara sahip modeller arasında doğrudan karşılaştırmak yanıltıcı olabilir (tokenizer verimliliği etkiler). BPC bu sorunu çözer.

In [ ]:
device

In [ ]:
def advanced_eval(model, tokenizer, text, label):
    inputs = tokenizer(text, return_tensors="pt").to(get_device(model))
    input_ids = inputs["input_ids"]
    char_count = len(text)
    token_count = input_ids.shape[1]
    
    with torch.no_grad():
        outputs = model(input_ids, labels=input_ids)
        loss = outputs.loss.item()
        ppl = np.exp(loss)
        bpc = (loss * token_count) / (char_count * np.log(2))
        
    return {"Model": label, "Loss": round(loss, 4), "PPL": round(ppl, 4), "BPC": round(bpc, 4)}

test_texts = [
    "Merkez Bankası faiz kararı piyasalarda dalgalanmaya neden oldu.",
    "Borsa İstanbul günü yüzde iki artışla kapattı.",
    "Enflasyon oranı beklentilerin üzerinde açıklandı.",
]

for text in test_texts:
    results = [advanced_eval(MODELS[name], TOKENIZERS[name], text, name) for name in MODELS]
    print(f"\nMetin: {text[:60]}...")
    display(pd.DataFrame(results))
clear_gpu()

## 2. Token Entropisi ve Güven Analizi

### Nedir?
Her token için iki şey ölçüyoruz:
- **Confidence (Güven):** Model, gerçek (doğru) token'a ne kadar olasılık atamış? `%90` diyorsa çok emin, `%2` diyorsa neredeyse rastgele tahmin etmiş demektir.
- **Entropy (Entropi):** Modelin olasılık dağılımının ne kadar "yayılmış" olduğu. Shannon entropisi ile ölçülür. Düşük entropi = model birkaç token'a odaklanmış (emin). Yüksek entropi = olasılık birçok token'a dağılmış (kararsız).

### Ne bulmayı bekliyoruz?
- **Fonksiyon kelimeleri** (de, ve, bir, için): Yüksek confidence, düşük entropi → bunlar bağlamdan kolayca tahmin edilebilir.
- **İçerik kelimeleri** (Ankara, teknoloji): Daha düşük confidence, daha yüksek entropi → birçok alternatif mümkün.
- İyi bir model, tahmin edilebilir yerlerde emin, belirsiz yerlerde dürüstçe kararsız olmalı.

### Nasıl yorumlarız?
- Bir token'da **yüksek confidence + düşük entropi** → model o bağlamı iyi öğrenmiş.
- **Düşük confidence + yüksek entropi** → model kararsız; ya bağlam yetersiz ya da model o pattern'i öğrenememiş.
- Tüm token'larda sürekli düşük confidence → model genel olarak zayıf eğitilmiş.

In [ ]:
def analyze_token_confidence(model, tokenizer, text):
    inputs = tokenizer(text, return_tensors="pt").to(get_device(model))
    input_ids = inputs["input_ids"]
    with torch.no_grad():
        logits = model(input_ids).logits[0, :-1, :]
        probs = F.softmax(logits, dim=-1)
        target_ids = input_ids[0, 1:]
        target_probs = probs[torch.arange(len(target_ids)), target_ids]
        entropy = -torch.sum(probs * torch.log(probs + 1e-10), dim=-1)
        
    #return entropy,target_probs
    return pd.DataFrame({
        "Token": [tokenizer.decode([t]) for t in target_ids],
        "Confidence (%)": (target_probs * 100).cpu().float().numpy().round(2),
        "Entropy": entropy.cpu().float().numpy().round(4)
  })

In [ ]:
text = "Merkez Bankası faiz oranını yüzde elli olarak belirledi."
for name in MODELS:
    print(f"\n{'='*50}\n{name}\n{'='*50}")
    display(analyze_token_confidence(MODELS[name], TOKENIZERS[name], text))
clear_gpu()

## 3. Logit Lens Görselleştirmesi

### Nedir?
Logit Lens, modelin **ara katmanlarına** bakarak bilginin nasıl adım adım inşa edildiğini gösteren bir tekniktir. Normalde sadece son katmanın çıktısını görürüz; logit lens ile her katmanın hidden state'ini alıp final norm + lm_head'den geçirerek "bu katman tek başına olsaydı ne tahmin ederdi?" sorusunu sorarız.

İki farklı bakış sunuyoruz:
- **Grafik (plot_logit_lens):** Hedef kelimenin (ör. "Ankara") olasılığının katmanlar boyunca nasıl arttığını gösterir.
- **Tablo (robust_logit_lens):** Her N katmanda modelin en yüksek olasılıklı tahminini listeler. Mimari bağımsız çalışır (Qwen/Llama/GPT-2).

### Ne bulmayı bekliyoruz?
- **İlk katmanlar:** Hedef kelime düşük olasılıkta olmalı — model henüz yüzeysel özellikler (n-gram, pozisyon) işliyor.
- **Orta katmanlar:** Olasılık yükselmeye başlamalı — sözdizimi ve anlamsal ilişkiler oluşuyor.
- **Son katmanlar:** Doğru cevabın olasılığı belirgin şekilde yüksek olmalı — bilgi kristalleşmiş.

### Nasıl yorumlarız?
- **Keskin yükseliş (sigmoid şeklinde):** Model bilgiyi belirli bir katman aralığında öğrenmiş → sağlıklı.
- **Erken yükseliş:** Model bu bilgiyi alt katmanlarda bile çözebiliyor → çok güçlü öğrenme.
- **Düz çizgi (yükselmiyor):** Model bu bilgiyi hiç öğrenememiş.
- **Modeller arası fark:** Hangi model bilgiyi daha erken katmanlarda çözüyor? Bu, o modelin o bilgiyi daha "derinden" içselleştirdiğini gösterir.

In [ ]:
def get_final_norm(model):
    """Modelin final norm katmanını dinamik olarak bul (Qwen/Llama vs GPT-2)."""
    if hasattr(model, 'model') and hasattr(model.model, 'norm'):
        return model.model.norm
    elif hasattr(model, 'transformer') and hasattr(model.transformer, 'ln_f'):
        return model.transformer.ln_f
    return None

def plot_logit_lens(model, tokenizer, text, target_word=" Ankara"):
    inputs = tokenizer(text, return_tensors="pt").to(get_device(model))
    target_id = tokenizer.encode(target_word)[-1]
    final_norm = get_final_norm(model)
    
    with torch.no_grad():
        outputs = model(inputs["input_ids"], output_hidden_states=True)
        hidden_states = outputs.hidden_states
        
    probs_list = []
    for h in hidden_states:
        norm_h = final_norm(h) if final_norm is not None else h
        logits = model.lm_head(norm_h)[0, -1, :]
        probs_list.append(F.softmax(logits, dim=-1)[target_id].item())
        
    plt.figure(figsize=(10, 4))
    plt.plot(probs_list, marker='o', color='teal', linewidth=2)
    plt.title(f"'{target_word}' Kelimesinin Katmanlar Boyunca Olasılığı")
    plt.xlabel("Layer")
    plt.ylabel("Probability")
    plt.grid(True, alpha=0.3)
    plt.show()

def robust_logit_lens(model, tokenizer, text, layer_step=4):
    inputs = tokenizer(text, return_tensors="pt").to(get_device(model))
    final_norm = get_final_norm(model)
    
    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True)
        hidden_states = outputs.hidden_states
        
    results = []
    for i in range(0, len(hidden_states), layer_step):
        state = final_norm(hidden_states[i]) if final_norm is not None else hidden_states[i]
        logits = model.lm_head(state)
        top_token = tokenizer.decode([torch.argmax(logits[0, -1, :]).item()])
        results.append({"Layer": i, "Predicted Next Token": top_token})
    return pd.DataFrame(results)

In [ ]:
text = "Türkiye'nin merkez bankasının adı "
for name in MODELS:
    print(f"\n--- {name}: Logit Lens (grafik) ---")
    plot_logit_lens(MODELS[name], TOKENIZERS[name], text, " TCMB")
    print(f"\n--- {name}: Robust Logit Lens (tablo) ---")
    display(robust_logit_lens(MODELS[name], TOKENIZERS[name], text))
    clear_gpu()

## 4. Mantıksal Surprisal (Şaşkınlık) Analizi

### Nedir?
Surprisal, modelin bir token'ı gördüğünde ne kadar "şaşırdığını" ölçer: `-log P(token | bağlam)`. Burada bunu **mantıksal tutarlılık testi** olarak kullanıyoruz: aynı bağlama mantıklı ve mantıksız bir devam verip, modelin hangisine daha çok şaşırdığını karşılaştırıyoruz.

**Örnek:**
- Bağlam: *"Hava çok sıcak olduğu için..."*
- Mantıklı devam: *"su aldım."* → Düşük surprisal beklenir
- Mantıksız devam: *"ateş aldım."* → Yüksek surprisal beklenir

### Ne bulmayı bekliyoruz?
- İyi bir model, mantıksız devama **belirgin şekilde daha yüksek** loss atamalı.
- "Fark (x)" sütunu bu oranı gösterir: 1.5x+ iyi, 2x+ çok iyi bir ayrım demektir.
- Fark 1.0'a yakınsa model mantıklı/mantıksız ayrımı yapamıyor → dünya bilgisi zayıf.

### Nasıl yorumlarız?
- **Yüksek fark (>1.5x):** Model neden-sonuç ilişkisini kavramış.
- **Düşük fark (~1.0x):** Model iki devamı eşit buluyor → bu bağlamdaki dünya bilgisi eksik.
- **Modeller arası:** Hangi model daha yüksek fark üretiyorsa, o model gerçek dünya ilişkilerini daha iyi öğrenmiş demektir.

In [ ]:
def compare_surprisal(model, tokenizer, context, logic_word, illogic_word, model_name=""):
    def get_last_loss(text):
        inputs = tokenizer(text, return_tensors="pt").to(get_device(model))
        with torch.no_grad():
            outputs = model(inputs["input_ids"], labels=inputs["input_ids"])
            logits, labels = outputs.logits[0, :-1, :], inputs["input_ids"][0, 1:]
            return F.cross_entropy(logits[-1:], labels[-1:]).item()

    l_ok, l_bad = get_last_loss(context + logic_word), get_last_loss(context + illogic_word)
    return {
        "Model": model_name,
        "Mantıklı": f"{logic_word} ({l_ok:.4f})",
        "Mantıksız": f"{illogic_word} ({l_bad:.4f})",
        "Fark (x)": round(l_bad / l_ok, 2)
    }

In [ ]:
context = "Enflasyon çok yükseldiği için merkez bankası faizi "
results = [
    compare_surprisal(MODELS[name], TOKENIZERS[name], context, " artırdı.", " düşürdü.", name)
    for name in MODELS
]
display(pd.DataFrame(results))
clear_gpu()

## 5. Multiple Choice (Ranking) Deneyi

### Nedir?
Modele bir soru ve birkaç seçenek verip, her seçeneğin **log-olasılık skorunu** hesaplıyoruz. En yüksek skoru alan seçenek, modelin "en olası" bulduğu cevaptır. Bu yöntem, base modelleri benchmark'larda değerlendirmenin standart yoludur (ör. MMLU, HellaSwag, ARC).

**Nasıl çalışır:**
1. `prompt + seçenek` birleştirilir
2. Seçenek token'larının log-olasılıkları toplanır
3. En yüksek toplam skoru alan seçenek "kazanır"

### Ne bulmayı bekliyoruz?
- Doğru cevap (ör. "Paris") en yüksek skora sahip olmalı.
- Skorlar arasındaki **fark** önemli: büyük fark = model çok emin, küçük fark = model kararsız.
- Yanlış cevabın birinci çıkması, modelin o bilgiyi öğrenmediğini gösterir.

### Nasıl yorumlarız?
- **Doğru cevap 1. sırada + büyük skor farkı:** Model bu bilgiyi sağlam öğrenmiş.
- **Doğru cevap 1. sırada ama fark küçük:** Model doğru biliyor ama tam emin değil.
- **Yanlış cevap 1. sırada:** Model bu konuda hatalı bilgi öğrenmiş veya hiç öğrenememiş.
- **Modeller arası:** Aynı soruda hangi model doğru cevabı daha yüksek farkla seçiyor?

In [ ]:
def rank_choices(model, tokenizer, prompt, choices):
    """Her seçenek için log-prob hesapla ve sırala.
    prefix_len bazlı — subword boundary shift sorununa karşı güvenli."""
    results = []
    prefix_len = tokenizer(prompt, return_tensors="pt")["input_ids"].shape[1]
    
    for choice in choices:
        full_text = prompt + choice
        inputs = tokenizer(full_text, return_tensors="pt").to(get_device(model))
        ids = inputs["input_ids"][0]
        n_choice_tokens = len(ids) - prefix_len
        
        with torch.no_grad():
            logits = model(inputs["input_ids"]).logits[0]
            log_probs = F.log_softmax(logits, dim=-1)
            
            total_lp = sum(log_probs[i-1, ids[i]].item() for i in range(prefix_len, len(ids)))
            norm_lp = total_lp / n_choice_tokens if n_choice_tokens > 0 else total_lp
            
        results.append({"Choice": choice, "Score (raw)": round(total_lp, 4), 
                        "Score (norm)": round(norm_lp, 4), "Tokens": n_choice_tokens})
    return pd.DataFrame(results).sort_values("Score (norm)", ascending=False)

In [ ]:
ranking_tests = [
    ("Türkiye'nin merkez bankasının adı nedir? Cevap:", [" TCMB", " FED", " ECB"], "TCMB"),
    ("Borsa İstanbul'un kısaltması:", [" BIST", " NYSE", " DAX"], "BIST"),
    ("Enflasyon artınca merkez bankası genellikle faizi", [" artırır.", " düşürür.", " sabit tutar."], "artırır."),
    ("Bir şirketin toplam varlıklarından borçları çıkarılınca kalan değere", [" özsermaye", " gelir", " gider"], "özsermaye"),
]

for name in MODELS:
    print(f"\n{'='*60}\n{name}\n{'='*60}")
    for prompt, choices, correct in ranking_tests:
        print(f"\n  Soru: {prompt} (Doğru: {correct})")
        display(rank_choices(MODELS[name], TOKENIZERS[name], prompt, choices))
clear_gpu()

## 6. Tokenizer Verimliliği (Fertility) Analizi

### Nedir?
Tokenizer fertility, bir metnin kaç token'a parçalandığını ve **karakter/token oranını** ölçer. Model ağırlıkları mükemmel olsa bile, kötü bir tokenizer modelin performansını ciddi şekilde düşürür — çünkü aynı anlam için daha fazla token harcamak demek, daha kısa context window ve daha yavaş inference demektir.

**Karakter/Token oranı** = Toplam karakter sayısı / Token sayısı

### Ne bulmayı bekliyoruz?
- Türkçe'ye özel eğitilmiş tokenizer'lar daha yüksek karakter/token oranı üretmeli (daha verimli).
- İngilizce ağırlıklı tokenizer'lar (ör. GPT-2) Türkçe'de çok sayıda token üretir — özellikle ekleri (lar, ler, daki, ndaki) ayrı ayrı tokenize eder.
- `First 10 Tokens` sütununda anlamlı alt kelimeler mi yoksa tek karakterler mi var? Bu tokenizer kalitesini gösterir.

### Nasıl yorumlarız?
| Chars/Token | Yorum |
|-------------|-------|
| > 4.0 | Çok verimli — Türkçe'ye iyi uyumlu |
| 2.5 - 4.0 | Orta — genel amaçlı tokenizer |
| < 2.5 | Verimsiz — Türkçe metni çok fazla parçalıyor |

> **Not:** Tokenizer verimliliği, PPL karşılaştırmalarını doğrudan etkiler. Verimsiz tokenizer → yapay olarak düşük PPL (token başına daha az bilgi taşıdığı için). BPC bunu normalize eder.

In [ ]:
def tokenizer_fertility_analysis(text, tokenizers_dict):
    results = []
    char_length = len(text)
    for name, tokenizer in tokenizers_dict.items():
        tokens = tokenizer.encode(text)
        token_count = len(tokens)
        chars_per_token = char_length / token_count if token_count > 0 else 0
        results.append({
            "Model": name,
            "Token Count": token_count,
            "Chars per Token": round(chars_per_token, 2),
            "First 10 Tokens": [tokenizer.decode([t]) for t in tokens][:10]
        })
    return pd.DataFrame(results)

In [ ]:
uzun_metin = "Merkez Bankası'nın faiz politikası, enflasyon beklentileri ve döviz kuru hareketleri piyasalardaki yatırımcı davranışını doğrudan etkilemektedir."
display(tokenizer_fertility_analysis(uzun_metin, TOKENIZERS))

## 7. Shannon Entropisi ve Top-K Güven Dağılımı

### Nedir?
Section 2'de entropi ve confidence'ı tablo olarak gördük. Burada aynı kavramı **görsel** olarak inceliyoruz: modelin son token pozisyonunda en yüksek olasılık verdiği K token'ı bar grafiğinde gösteriyoruz. Başlıkta Shannon entropisi de yer alır.

Bu görselleştirme, modelin "kafasının içine bakmak" gibidir — tek bir tahmin anında karar mekanizmasını gösterir.

### Ne bulmayı bekliyoruz?
- **Emin model:** 1-2 token çok yüksek olasılıkta, geri kalanı neredeyse sıfır. Entropi düşük.
- **Kararsız model:** Olasılık birçok token'a yayılmış. Entropi yüksek.
- Bağlam çok belirleyiciyse (ör. "Türkiye'nin Başkenti:") iyi modelin dominant bir peak göstermesini bekliyoruz.

### Nasıl yorumlarız?
- **Tek dominant çubuk:** Model çok emin → bu bağlamda güçlü öğrenme.
- **Birden fazla yakın çubuk:** Model birkaç alternatif arasında kararsız.
- **Düz dağılım:** Model neredeyse rastgele tahmin ediyor → çok zayıf.
- **Modeller arası:** Aynı bağlamda hangi model daha "sivri" (confident) bir dağılım üretiyor?

In [ ]:
def plot_top_k_confidence(model, tokenizer, text, k=5, model_name="Model"):
    inputs = tokenizer(text, return_tensors="pt").to(get_device(model))
    with torch.no_grad():
        logits = model(**inputs).logits[0, -1, :]
    probs = F.softmax(logits, dim=-1)
    entropy = -torch.sum(probs * torch.log(probs + 1e-9)).item()
    vals, idxs = torch.topk(probs, k)
    
    plt.figure(figsize=(10, 4))
    sns.barplot(x=vals.cpu().float().numpy()*100, y=[tokenizer.decode([i]) for i in idxs], palette="viridis")
    plt.title(f"{model_name} - Top {k} Confidence | Entropy: {entropy:.4f}\nContext: '{text}'")
    plt.xlabel("Probability (%)")
    plt.show()



In [ ]:
text = "Borsa İstanbul'un kısaltması : "
for name in MODELS:
    plot_top_k_confidence(MODELS[name], TOKENIZERS[name], text, k=10, model_name=name)
    clear_gpu()

## 8. Sliding Window ile Gerçek Perplexity

### Nedir?
Section 1'deki PPL kısa metinler için çalışır, ama uzun metinlerde modelin **context window sınırına** takılırız. Sliding window yöntemi, metni belirli bir `stride` ile kayan pencereler halinde tarar ve her penceredeki loss'u toplayarak **tüm metin üzerinde doğru bir PPL** hesaplar.

**Nasıl çalışır:**
1. Metin `max_length` boyutunda pencerelere bölünür
2. Her pencere `stride` kadar kaydırılır (overlap ile)
3. Sadece yeni görülen token'ların loss'u hesaplanır (önceki overlap kısmı -100 ile maskelenir)
4. Tüm loss'lar ortalalanıp `exp()` alınır → gerçek PPL

### Ne bulmayı bekliyoruz?
- Sliding window PPL, kısa metin PPL'ine yakın ama biraz farklı olabilir — çünkü uzun bağlam kullanır.
- İyi model: tekrar eden metinlerde (aynı cümle 10 kez) bile düşük PPL → pattern'i yakalamış.
- Kötü model: PPL yüksek kalır çünkü bağlam uzadıkça bilgiyi tutamıyor.

### Nasıl yorumlarız?
- Section 1 PPL'i ile karşılaştırın: çok farklıysa modelin uzun bağlam yönetimi zayıf olabilir.
- Modeller arası fark, özellikle uzun metin senaryolarında hangi modelin daha tutarlı olduğunu gösterir.

In [ ]:
def calculate_sliding_window_ppl(model, tokenizer, text, stride=512, max_length=1024):
    encodings = tokenizer(text, return_tensors="pt")
    seq_len = encodings.input_ids.size(1)
    nlls = []
    prev_end_loc = 0
    for begin_loc in range(0, seq_len, stride):
        end_loc = min(begin_loc + max_length, seq_len)
        trg_len = end_loc - prev_end_loc
        input_ids = encodings.input_ids[:, begin_loc:end_loc].to(get_device(model))
        target_ids = input_ids.clone()
        target_ids[:, :-trg_len] = -100
        with torch.no_grad():
            outputs = model(input_ids, labels=target_ids)
            nlls.append(outputs.loss)
        prev_end_loc = end_loc
        if end_loc == seq_len: break
    return torch.exp(torch.stack(nlls).mean()).item()



In [ ]:
uzun_test_metni = " " + " ".join(["Finansal piyasalarda volatilite artmaya devam ediyor."] * 10)

results = []
for name in MODELS:
    ppl = calculate_sliding_window_ppl(MODELS[name], TOKENIZERS[name], uzun_test_metni)
    results.append({"Model": name, "Sliding Window PPL": round(ppl, 4)})
    clear_gpu()
display(pd.DataFrame(results))

## 9. Attention Map (Isı Haritası) Görselleştirmesi

### Nedir?
Attention map, transformer'ın **self-attention mekanizmasını** görselleştirir. Her token'ın diğer token'lara ne kadar "dikkat ettiğini" bir ısı haritasında gösterir. Satırlar "bakan" token, sütunlar "bakılan" token'dır. Parlak hücreler güçlü dikkat bağlantılarını temsil eder.

Burada modelin **son katmanının ilk head'ini** gösteriyoruz — bu genellikle en "anlamsal" dikkat örüntülerini barındırır.

### Ne bulmayı bekliyoruz?
- **Diyagonal çizgi:** Her token kendisine ve bir önceki token'a dikkat eder → temel autoregressive pattern.
- **Dikey çizgiler:** Belirli bir token'a birçok token dikkat eder → bu "önemli" bir kelime (ör. özne, fiil).
- **Blok yapılar:** Birbirine yakın token'lar grup halinde birbirine dikkat eder → sözdizimsel yapı (cümlecik sınırları).

### Nasıl yorumlarız?
- **"Ankara" kelimesine yoğun dikkat:** Model, başkent bilgisini bu token üzerinden çözüyor.
- **Başlangıç token'ına (BOS/ilk token) dikkat:** Birçok modelde "bilgi havuzu" olarak kullanılır — attention sink.
- **Dağınık dikkat:** Head o katmanda belirgin bir pattern öğrenememiş.
- **Modeller arası:** Aynı metin için farklı modellerin dikkat örüntülerini karşılaştırarak hangi modelin daha yapısal dikkat geliştirdiğini görebilirsiniz.

> **Not:** Sadece `attn_implementation="eager"` ile yüklenen modeller attention çıktısı verir. SDPA kullanan modellerde bu özellik desteklenmeyebilir.

In [ ]:
def plot_attention_map(model, tokenizer, text, layer=-1, head=0, model_name=""):
    inputs = tokenizer(text, return_tensors="pt").to(get_device(model))
    tokens = [tokenizer.decode([t]) for t in inputs["input_ids"][0]]
    
    with torch.no_grad():
        outputs = model(**inputs, output_attentions=True)
        
    attentions = outputs.attentions
    if attentions is None:
        print(f"{model_name}: Attention çıktısı üretmedi (output_attentions desteklenmiyor olabilir).")
        return

    valid_attns = [a for a in attentions if a is not None]
    if not valid_attns:
        print(f"{model_name}: Görselleştirilebilecek Full Attention katmanı bulunamadı.")
        return

    if abs(layer) >= len(valid_attns):
        print(f"{model_name}: Geçersiz katman! Sadece {len(valid_attns)} Full Attention katmanı mevcut.")
        return
    
    attn_matrix = valid_attns[layer][0, head, :, :].float().cpu().numpy()
    
    plt.figure(figsize=(10, 8))
    sns.heatmap(attn_matrix, xticklabels=tokens, yticklabels=tokens, cmap="viridis", square=True)
    plt.title(f"{model_name} - Attention Map (Layer {layer}, Head {head})")
    plt.xticks(rotation=45)
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()

In [ ]:
text = "Merkez Bankası faiz oranını artırdı."
for name in MODELS:
    plot_attention_map(MODELS[name], TOKENIZERS[name], text, layer=-1, head=0, model_name=name)
    clear_gpu()

---

# Katman 3: Linguistic Competence

---

## 10. Subject-Verb Agreement Testi

### Nedir?
Model **hiyerarşik sözdizimi** öğrenmiş mi? Bunu test etmenin klasik yolu: özne ile fiil arasına "attractor" (yanıltıcı) isimler koyarak modelin doğru çekimi (tekil/çoğul) seçip seçemediğine bakmaktır.

**Örnek:**
- *"The keys to the cabinet..."* → `are` (doğru, özne "keys" çoğul) vs `is` (yanlış, "cabinet" tekil ama özne değil)
- Model "cabinet"e aldanırsa → yüzeysel proximity heuristic kullanıyor, gerçek syntax öğrenememiş.

**Ref:** Linzen et al. (2016), Goldberg (2019), Marvin & Linzen (2018)

### Ne bulmayı bekliyoruz?
- **Easy (1 attractor):** 1B+ model %90+ geçmeli.
- **Hard (center-embedding):** Sadece güçlü modeller %50+ yapabilir. Küçük modelin başarısız olması normal.
- Difficulty breakdown çok önemli: Easy iyi + Hard kötü = surface pattern matching.

### Nasıl yorumlarız?
| Durum | Yorum |
|-------|-------|
| Easy ✅ Hard ✅ | Güçlü hierarchical syntax |
| Easy ✅ Hard ❌ | Surface pattern OK, deep structure zayıf |
| Easy ❌ | Temel seviyede sorun — model boyutu/data yetersiz |

In [ ]:
def run_agreement_tests(model, tokenizer, model_name=""):
    """Özne-yüklem uyumu testleri — finans domain, Türkçe."""
    tests = [
        ("Bankanın şubelerindeki müdürler", " toplantıya katıldı.", " toplantıya katıldım.",
         "easy", "3. çoğul kişi uyumu"),
        ("Yatırımcıların portföyündeki hisse senetleri", " değer kazandı.", " değer kazandım.",
         "easy", "Attractor 'portföy' tekil, özne 'senetleri' çoğul"),
        ("Şirketin denetçilerinin hazırladığı rapor", " onaylandı.", " onaylandılar.",
         "medium", "İç içe tamlama — özne 'rapor' tekil"),
        ("Fonun yöneticisinin danıştığı analistler", " uyardı.", " uyardım.",
         "hard", "2 seviye iç içe tamlama"),
        ("Merkez bankasının açıkladığı verilere göre piyasalar", " düştü.", " düştüm.",
         "hard", "Uzun bağımlılık — özne 'piyasalar'"),
    ]

    results = []
    for prefix, correct, wrong, diff, note in tests:
        lp_c = get_logprob(model, tokenizer, prefix, correct)
        lp_w = get_logprob(model, tokenizer, prefix, wrong)
        passed = lp_c > lp_w
        results.append({
            "Prefix": prefix[:40] + "...",
            "Difficulty": diff,
            "Passed": "PASS" if passed else "FAIL",
            "Margin": round(lp_c - lp_w, 3),
            "Note": note
        })

    df = pd.DataFrame(results)
    total = sum(1 for r in results if r["Passed"] == "PASS")
    print(f"  {model_name}: {total}/{len(results)}")
    return df

In [ ]:
for name in MODELS:
    print(f"\n{'='*60}\n{name}\n{'='*60}")
    display(run_agreement_tests(MODELS[name], TOKENIZERS[name], name))
clear_gpu()

## 11. Word Order (Kelime Sırası) Testi

### Nedir?
Grammatik doğru bir cümle ile aynı kelimelerin rastgele sıralandığı versiyonu karşılaştırılır. Model kelime sırasının önemini anlamışsa, doğru cümleye **belirgin şekilde düşük PPL** atamalıdır.

### Ne bulmayı bekliyoruz?
- PPL oranı (bozuk/doğru) > 5x: çok duyarlı — iyi.
- 2-5x: makul.
- < 2x: kelime sırası bilgisi zayıf.

In [ ]:
def run_word_order_tests(model, tokenizer, model_name=""):
    pairs = [
        ("Merkez Bankası faiz oranını artırdı.", "Artırdı oranını faiz Bankası Merkez."),
        ("Yatırımcılar borsadan hisse senedi aldı.", "Aldı senedi hisse borsadan yatırımcılar."),
        ("Şirket bilançosunu kamuoyuyla paylaştı.", "Paylaştı kamuoyuyla bilançosunu şirket."),
        ("Dolar kuru dün akşam rekor kırdı.", "Kırdı rekor akşam dün kuru dolar."),
    ]
    
    results = []
    for good, bad in pairs:
        inputs_g = tokenizer(good, return_tensors="pt").to(get_device(model))
        inputs_b = tokenizer(bad, return_tensors="pt").to(get_device(model))
        with torch.no_grad():
            loss_g = model(**inputs_g, labels=inputs_g["input_ids"]).loss.item()
            loss_b = model(**inputs_b, labels=inputs_b["input_ids"]).loss.item()
        ppl_g, ppl_b = math.exp(loss_g), math.exp(loss_b)
        ratio = ppl_b / ppl_g if ppl_g > 0 else 0
        results.append({
            "Doğru": good, "Bozuk": bad,
            "PPL (doğru)": round(ppl_g, 1), "PPL (bozuk)": round(ppl_b, 1),
            "Oran": round(ratio, 1), "Passed": "PASS" if ppl_g < ppl_b else "FAIL"
        })
    return pd.DataFrame(results)

In [ ]:
for name in MODELS:
    print(f"\n{'='*60}\n{name}\n{'='*60}")
    display(run_word_order_tests(MODELS[name], TOKENIZERS[name], name))
clear_gpu()

## 12. Türkçe Morfoloji: Ünlü Uyumu ve Kişi Uyumu

### Nedir?
Türkçe agglutinative (eklemeli) bir dil. İki temel morfolojik kuralı test ediyoruz:

**Ünlü Uyumu:** Eklerin ünlüsü, kök kelimenin son ünlüsüne uyar.
- Kalın ünlü (a, ı, o, u) → kalın ek: *kitap**lar*** (doğru), *kitap**ler*** (yanlış)
- İnce ünlü (e, i, ö, ü) → ince ek: *ev**ler*** (doğru), *ev**lar*** (yanlış)

**Kişi Uyumu:** Fiil çekimi özneyle uyumlu olmalı.
- *Ali okula gitti.* (3. tekil) vs *Ali okula gittim.* (1. tekil — yanlış)

**Ref:** Oflazer (1994), Rust et al. (2021) "How Good Is Your Tokenizer?"

### Ne bulmayı bekliyoruz?
- Türkçe ağırlıklı eğitilmiş modeller (Kumru, Turkish-GPT2) burada güçlü olmalı.
- Çok dilli ama Türkçe'de zayıf modeller (Qwen) ünlü uyumunda başarısız olabilir.
- Tokenizer fertility burada doğrudan etkili: eki ayrı token olarak görebilen tokenizer avantajlı.

In [ ]:
def run_turkish_morphology_tests(model, tokenizer, model_name=""):
    """Türkçe ünlü uyumu ve kişi uyumu testleri — finans domain."""
    
    vowel_tests = [
        ("Faizler", "den", "dan", "Front: e→e (faiz+ler+den)"),
        ("Bankalar", "dan", "den", "Back: a→a (banka+lar+dan)"),
        ("Borsa", "lar", "ler", "Back plural (borsa+lar)"),
        ("Getiri", "ler", "lar", "Front plural (getiri+ler)"),
        ("Portföy", "leri", "ları", "Front possessive (portföy+leri)"),
        ("Yatırım", "ları", "leri", "Back possessive (yatırım+ları)"),
    ]
    
    case_tests = [
        ("Analist raporu", " yayımladı.", " yayımladım.", "3sg vs 1sg"),
        ("Yatırımcılar hisse", " sattı.", " sattın.", "3pl vs 2sg"),
        ("Dün akşam piyasalar", " kapandı.", " kapanacak.", "Temporal: dün→geçmiş zaman"),
        ("Ben bu hisseyi", " aldım.", " aldı.", "1sg vs 3sg"),
    ]
    
    results = []
    
    for prefix, correct, wrong, note in vowel_tests:
        lp_c = get_logprob(model, tokenizer, prefix, correct)
        lp_w = get_logprob(model, tokenizer, prefix, wrong)
        results.append({
            "Test": "Ünlü Uyumu", "Prefix": prefix,
            "Doğru": correct, "Yanlış": wrong,
            "Passed": "PASS" if lp_c > lp_w else "FAIL",
            "Margin": round(lp_c - lp_w, 3), "Note": note
        })
    
    for prefix, correct, wrong, note in case_tests:
        lp_c = get_logprob(model, tokenizer, prefix, correct)
        lp_w = get_logprob(model, tokenizer, prefix, wrong)
        results.append({
            "Test": "Kişi Uyumu", "Prefix": prefix,
            "Doğru": correct, "Yanlış": wrong,
            "Passed": "PASS" if lp_c > lp_w else "FAIL",
            "Margin": round(lp_c - lp_w, 3), "Note": note
        })
    
    df = pd.DataFrame(results)
    vowel_pass = sum(1 for r in results if r["Test"] == "Ünlü Uyumu" and r["Passed"] == "PASS")
    case_pass = sum(1 for r in results if r["Test"] == "Kişi Uyumu" and r["Passed"] == "PASS")
    print(f"  {model_name}: Ünlü Uyumu {vowel_pass}/{len(vowel_tests)}, Kişi Uyumu {case_pass}/{len(case_tests)}")
    return df

In [ ]:
for name in MODELS:
    print(f"\n{'='*60}\n{name}\n{'='*60}")
    display(run_turkish_morphology_tests(MODELS[name], TOKENIZERS[name], name))
clear_gpu()

---

# Katman 4: Factual Knowledge

---

## 13. Factual Knowledge: Popular vs Rare

### Nedir?
Model parametrelerine ne kadar **dünya bilgisi** encode etmiş? Bunu iki düzeyde test ediyoruz:

- **Popular (sık) bilgi:** "Fransa'nın başkenti Paris" — eğitim verisinde çok geçer, çoğu model bilir.
- **Rare (nadir) bilgi:** "Burkina Faso'nun başkenti Ouagadougou" — çok az geçer, sadece iyi eğitilmiş büyük modeller bilir.

**Contrastive Margin:** Doğru cevap ile yanlış cevap arasındaki log-prob farkı. Margin > 0 = model doğruyu biliyor. Büyük margin = daha emin.

**Ref:** Petroni et al. (2019) LAMA, Chang et al. (2024) learnability threshold, Dai et al. (2022) Knowledge Neurons

### Ne bulmayı bekliyoruz?
- Popular bilgide tüm modeller başarılı olmalı.
- Rare bilgide sadece büyük ve iyi eğitilmiş modeller başarılı.
- Chang et al. (2024) bulgusu: "unpopular" bilgi learnability threshold altında kalırsa **asla öğrenilemez**.
- Contrastive margin, doğru/yanlış binary'sinden daha bilgilendirici — modelin ne kadar emin olduğunu gösterir.

### Nasıl yorumlarız?
| Popular | Rare | Yorum |
|---------|------|-------|
| ✅ Yüksek margin | ✅ | Güçlü bilgi kapasitesi |
| ✅ Yüksek margin | ❌ | Normal — model boyutu/data sınırlı |
| ❌ | ❌ | Ciddi sorun — data coverage veya training yetersiz |
| ❌ | ✅ | Garip — muhtemelen data contamination |

In [ ]:
def contrastive_margin(model, tokenizer, prompt, correct, incorrect):
    """Doğru vs yanlış cevap arasındaki log-prob farkı."""
    lp_c = get_logprob(model, tokenizer, prompt, correct)
    lp_w = get_logprob(model, tokenizer, prompt, incorrect)
    return {"margin": round(lp_c - lp_w, 3), "passed": lp_c > lp_w}


def run_factual_tests(model, tokenizer, model_name=""):
    """Popular vs rare finansal bilgi testi."""
    tests = [
        ("Türkiye'nin merkez bankasının kısaltması", " TCMB", " FED", "popular"),
        ("Borsa İstanbul'un kısaltması", " BIST", " NYSE", "popular"),
        ("Türkiye'nin para birimi", " Türk lirası", " dolar", "popular"),
        ("Enflasyon hesaplanırken kullanılan endeksin adı", " TÜFE", " BİST", "popular"),
        ("Şirketlerin finansal durumunu gösteren tabloya", " bilanço", " fatura", "medium"),
        ("Tahvil fiyatı yükseldiğinde getirisi", " düşer", " artar", "medium"),
        ("Black-Scholes modeli hangi finansal enstrümanın fiyatlamasında kullanılır?", " Opsiyon", " Tahvil", "rare"),
        ("Basel III düzenlemesinde bankaların asgari sermaye yeterlilik oranı", " yüzde sekiz", " yüzde iki", "rare"),
    ]
    
    results = []
    for prompt, correct, incorrect, diff in tests:
        r = contrastive_margin(model, tokenizer, prompt, correct, incorrect)
        results.append({
            "Prompt": prompt[:45], "Doğru": correct, "Yanlış": incorrect,
            "Difficulty": diff, "Margin": r["margin"],
            "Passed": "PASS" if r["passed"] else "FAIL"
        })
    
    df = pd.DataFrame(results)
    for diff in ["popular", "medium", "rare"]:
        subset = [r for r in results if r["Difficulty"] == diff]
        passed = sum(1 for r in subset if r["Passed"] == "PASS")
        print(f"  {model_name} [{diff}]: {passed}/{len(subset)}")
    return df

In [ ]:
for name in MODELS:
    print(f"\n{'='*60}\n{name}\n{'='*60}")
    display(run_factual_tests(MODELS[name], TOKENIZERS[name], name))
clear_gpu()

---

# Katman 5: Reasoning & In-Context Learning

---

## 14. ICL (In-Context Learning) Curve

### Nedir?
Base modelin en önemli emergent ability'lerinden biri: prompt'ta verilen örneklerden pattern çıkarıp uygulama. **ICL curve**, 0-shot'tan N-shot'a kadar modelin hedef cevaba olan loss'unun nasıl düştüğünü gösterir.

**Nasıl çalışır:**
1. 0-shot: sadece soru → loss yüksek
2. 1-shot: 1 örnek + soru → loss düşmeye başlar
3. N-shot: N örnek + soru → her örnekle loss düşmeli

**Ref:** Brown et al. (2020) GPT-3, Garg et al. (2022)

### Ne bulmayı bekliyoruz?
- Her eklenen örnekle loss düşmeli — bu ICL'in çalıştığını gösterir.
- Düşüş hızı modelin ICL kapasitesini yansıtır.
- Küçük modellerde düşüş yavaş veya hiç olmayabilir — bu normal.

### Nasıl yorumlarız?
- **Keskin düşüş (1-2 shot'ta):** Model pattern'i hızlı kavrıyor → güçlü ICL.
- **Gradüel düşüş:** Öğreniyor ama yavaş.
- **Düz çizgi:** ICL çalışmıyor — model örneklerden öğrenemiyor.

In [ ]:
def evaluate_icl_curve(model, tokenizer, base_prompt, examples,
                       target_question, target_answer, model_name=""):
    """0-shot'tan N-shot'a kadar target loss değişimi."""
    results = []
    current_prompt = base_prompt

    for k in range(len(examples) + 1):
        if k > 0:
            current_prompt += examples[k-1] + "\n"
        
        full = current_prompt + target_question + target_answer
        inputs = tokenizer(full, return_tensors="pt").to(get_device(model))
        target_ids = tokenizer.encode(target_answer, add_special_tokens=False)
        target_len = len(target_ids)

        with torch.no_grad():
            logits = model(inputs["input_ids"]).logits[0]
            logits_slice = logits[-(target_len+1):-1, :]
            log_probs = F.log_softmax(logits_slice, dim=-1)
            score = log_probs[torch.arange(target_len),
                              torch.tensor(target_ids).to(get_device(model))].mean().item()

        results.append({"k_shot": k, "target_loss": round(-score, 4)})
    
    return pd.DataFrame(results)


def plot_icl_curves(models_dict, tokenizers_dict, base_prompt, examples,
                    target_question, target_answer):
    """Tüm modellerin ICL curve'lerini tek grafikte göster."""
    plt.figure(figsize=(10, 5))
    
    for name in models_dict:
        df = evaluate_icl_curve(models_dict[name], tokenizers_dict[name],
                                base_prompt, examples, target_question,
                                target_answer, name)
        plt.plot(df["k_shot"], df["target_loss"], marker='o', label=name, linewidth=2)
    
    plt.xlabel("Number of Examples (k-shot)")
    plt.ylabel("Target Loss (düşük = daha iyi)")
    plt.title("ICL Learning Curve")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

In [ ]:
# Finansal terim çevirisi ICL testi
base = "Finansal terimlerin Türkçe karşılıkları:\n"
examples = ["Equity -> Özsermaye", "Bond -> Tahvil", "Inflation -> Enflasyon", "Dividend -> Temettü"]
target_q = "Hedge -> "
target_a = "Korunma"

plot_icl_curves(MODELS, TOKENIZERS, base, examples, target_q, target_a)
clear_gpu()

## 15. Flipped Labels Testi (Gerçek ICL mi, Semantic Prior mı?)

### Nedir?
ICL'in en kritik testi. Normal label'lar ile ters çevrilmiş label'lar karşılaştırılır:

- **Normal:** *"Great! → Positive, Terrible → Negative, Amazing → ?"* → Positive (kolay — semantic prior ile uyumlu)
- **Flipped:** *"Great! → Negative, Terrible → Positive, Amazing → ?"* → Negative (ZOR — label'lar ters)

Eğer model flipped'da da "Negative" derse → **gerçek ICL** (context'teki pattern'i takip ediyor).
Eğer flipped'da "Positive" derse → **semantic prior baskın**, ICL zayıf.

**Ref:** Wei et al. (2023) "Larger LMs Do In-Context Learning Differently"

### Ne bulmayı bekliyoruz?
- **Küçük modeller (<3B):** Flipped'da başarısız olmaları **normal ve beklenen**. Semantic prior baskın.
- **Büyük modeller (7B+):** Flipped'da da pattern'i takip etmeleri beklenir.
- Bu test, model boyutunun ICL kapasitesine etkisini net gösterir.

### Nasıl yorumlarız?
| Normal | Flipped | Yorum |
|--------|---------|-------|
| ✅ | ✅ | Gerçek ICL — context'ten öğreniyor |
| ✅ | ❌ | Semantic prior baskın — ICL zayıf (küçük modelde beklenen) |
| ❌ | ❌ | ICL çalışmıyor |

In [ ]:
def test_flipped_labels(model, tokenizer, model_name=""):
    """Normal vs flipped label ICL testi — finans sentiment."""
    
    normal_prompt = "Faiz artırıldı. -> Negatif\nEnflasyon düştü. -> Pozitif\nİşsizlik arttı. ->"
    flipped_prompt = "Faiz artırıldı. -> Pozitif\nEnflasyon düştü. -> Negatif\nİşsizlik arttı. ->"
    
    lp_pos_normal = get_logprob(model, tokenizer, normal_prompt, " Pozitif")
    lp_neg_normal = get_logprob(model, tokenizer, normal_prompt, " Negatif")
    normal_ok = lp_neg_normal > lp_pos_normal  # İşsizlik arttı -> Negatif bekliyoruz
    
    lp_pos_flipped = get_logprob(model, tokenizer, flipped_prompt, " Pozitif")
    lp_neg_flipped = get_logprob(model, tokenizer, flipped_prompt, " Negatif")
    flipped_ok = lp_pos_flipped > lp_neg_flipped  # Flipped'da Pozitif bekliyoruz
    
    if normal_ok and flipped_ok:
        verdict = "GERÇEK ICL — pattern'i takip ediyor"
    elif normal_ok and not flipped_ok:
        verdict = "SEMANTIC PRIOR baskın — ICL zayıf (küçük modelde beklenen)"
    else:
        verdict = "ICL çalışmıyor"
    
    return {
        "Model": model_name,
        "Normal": "PASS" if normal_ok else "FAIL",
        "Normal Margin": round(lp_neg_normal - lp_pos_normal, 3),
        "Flipped": "PASS" if flipped_ok else "FAIL",
        "Flipped Margin": round(lp_pos_flipped - lp_neg_flipped, 3),
        "Verdict": verdict
    }

In [ ]:
results = [test_flipped_labels(MODELS[name], TOKENIZERS[name], name) for name in MODELS]
display(pd.DataFrame(results))
clear_gpu()

---

# Katman 6: Generation Quality & Calibration

---

## 16. Generation Quality: Repetition ve Diversity

### Nedir?
Model uzun text ürettiğinde ne kadar tekrarlı (repetitive) ve ne kadar çeşitli (diverse)? İki temel metrik:

- **Rep-4:** 4-gram tekrar oranı. Üretilen text'te aynı 4-kelimelik dizinin kaç kez tekrarlandığı. `1 - (unique 4-grams / total 4-grams)`.
- **Distinct-2:** Unique 2-gram oranı. Yüksek = çeşitli text, düşük = tekdüze.

**Ref:** Holtzman et al. (2020) "Neural Text Degeneration", Su et al. (2022) SimCTG

### Ne bulmayı bekliyoruz?
- İyi model: düşük tekrar, yüksek çeşitlilik.
- Kötü model: aynı cümle/pattern defalarca tekrarlanır (degeneration).

### Nasıl yorumlarız?
| Metrik | İyi | Alarm | Ciddi Sorun |
|--------|-----|-------|-------------|
| Rep-4 | < 5% | 5-15% | > 15% |
| Distinct-2 | > 0.8 | 0.5-0.8 | < 0.5 |

In [ ]:
def generation_metrics(text):
    """Rep-4 ve Distinct-2 hesapla."""
    words = text.split()
    n = len(words)
    if n < 4:
        return {"rep_4": 0.0, "distinct_2": 0.0, "word_count": n}
    fours = [tuple(words[i:i+4]) for i in range(n-3)]
    twos = [tuple(words[i:i+2]) for i in range(n-1)]
    return {
        "rep_4": round(1 - len(set(fours)) / len(fours), 4),
        "distinct_2": round(len(set(twos)) / len(twos), 4),
        "word_count": n
    }


def run_generation_quality(models_dict, tokenizers_dict, prompts):
    """Birden fazla prompt ile generation quality ölç."""
    results = []
    for name in models_dict:
        all_rep4, all_dist2 = [], []
        for prompt in prompts:
            text = sampled_generate(models_dict[name], tokenizers_dict[name], prompt, max_tokens=150)
            m = generation_metrics(text)
            all_rep4.append(m["rep_4"])
            all_dist2.append(m["distinct_2"])
        
        avg_rep4 = round(np.mean(all_rep4), 4)
        avg_dist2 = round(np.mean(all_dist2), 4)
        results.append({
            "Model": name,
            "Rep-4 (avg)": avg_rep4,
            "Distinct-2 (avg)": avg_dist2,
            "Rep-4 Status": "OK" if avg_rep4 < 0.05 else ("ALARM" if avg_rep4 < 0.15 else "SORUN"),
            "Dist-2 Status": "OK" if avg_dist2 > 0.8 else ("ALARM" if avg_dist2 > 0.5 else "SORUN"),
        })
    return pd.DataFrame(results)

In [ ]:
prompts = [
    "Merkez Bankası'nın faiz kararının ardından piyasalarda",
    "Türkiye ekonomisinde son çeyrekte büyüme",
    "Borsa İstanbul'da bugün en çok yükselen sektör",
]

display(run_generation_quality(MODELS, TOKENIZERS, prompts))
clear_gpu()

## 17. Calibration: ECE ve Brier Score

### Nedir?
Modelin **güveni (confidence)** ile **gerçek doğruluğu (accuracy)** orantılı mı? Model "%90 eminim" dediğinde gerçekten %90 oranında doğru mu?

- **ECE (Expected Calibration Error):** Confidence bin'leri oluşturup her bin'de `|accuracy - confidence|` farkının ağırlıklı ortalaması. 0 = mükemmel kalibrasyon.
- **Brier Score:** Her tahmin için `(1 - P(doğru token))²`. Hem doğruluğu hem kalibrasyonu ölçer.

### Ne bulmayı bekliyoruz?
- İyi kalibre model: ECE < 0.05
- Overconfident model: ECE > 0.15 — model emin olduğu yerlerde yanılıyor. Fine-tuning sonrası sorun yaratır.

### Nasıl yorumlarız?
| ECE | Yorum |
|-----|-------|
| < 0.05 | İyi kalibre |
| 0.05-0.15 | Makul |
| > 0.15 | Overconfident — post-training calibration gerekebilir |

In [ ]:
def compute_calibration(model, tokenizer, texts, num_bins=10):
    """ECE ve Brier Score hesapla."""
    all_confidences = []
    all_accuracies = []
    brier_scores = []

    for text in texts:
        inputs = tokenizer(text, return_tensors="pt").to(get_device(model))
        ids = inputs["input_ids"][0]
        with torch.no_grad():
            logits = model(**inputs).logits[0]

        probs = F.softmax(logits[:-1], dim=-1)
        targets = ids[1:]

        confidences, predictions = torch.max(probs, dim=-1)
        correct = (predictions == targets).float()

        target_probs = probs[torch.arange(len(targets)), targets]
        brier = torch.mean((1.0 - target_probs) ** 2).item()
        brier_scores.append(brier)

        all_confidences.extend(confidences.cpu().float().tolist())
        all_accuracies.extend(correct.cpu().tolist())

    # ECE
    confs = np.array(all_confidences)
    accs = np.array(all_accuracies)
    ece = 0.0
    bins = np.linspace(0, 1, num_bins + 1)
    for i in range(num_bins):
        mask = (confs > bins[i]) & (confs <= bins[i+1])
        if mask.sum() > 0:
            ece += np.abs(accs[mask].mean() - confs[mask].mean()) * mask.mean()

    return {"ECE": round(ece, 4), "Brier Score": round(np.mean(brier_scores), 4)}

In [ ]:
cal_texts = [
    "Merkez Bankası faiz kararı piyasalarda dalgalanmaya neden oldu.",
    "Borsa İstanbul günü yüzde iki artışla kapattı.",
    "Enflasyon oranı beklentilerin üzerinde açıklandı.",
    "Şirketin yıllık geliri bir önceki yıla göre yüzde on beş arttı.",
    "Dolar kuru son bir ayda yüzde beş değer kazandı.",
]

results = []
for name in MODELS:
    cal = compute_calibration(MODELS[name], TOKENIZERS[name], cal_texts)
    cal["Model"] = name
    results.append(cal)
display(pd.DataFrame(results)[["Model", "ECE", "Brier Score"]])
clear_gpu()

## 18. Paraphrase Consistency (Bilgi Sağlamlığı)

### Nedir?
Aynı soru farklı formda sorulunca model **aynı cevabı** mı veriyor? Tutarsızlık, bilginin **fragile** (kırılgan) encode edildiğini gösterir — küçük prompt değişikliklerinde farklı cevap = bilgi yüzeysel.

**Ref:** Elazar et al. (2021) "Measuring and Improving Consistency in PLMs"

### Ne bulmayı bekliyoruz?
- İyi model: 3 farklı formülasyona aynı cevap (veya aynı log-prob sıralaması).
- Kötü model: her formda farklı cevap → bilgi robust değil.

### Nasıl yorumlarız?
- **Tutarlı (3/3):** Bilgi sağlam encode edilmiş.
- **Kısmen tutarlı (2/3):** Formülasyona duyarlı — bazı pattern'lerde güçlü, bazılarında zayıf.
- **Tutarsız (1/3 veya 0/3):** Bilgi fragile — fine-tuning'de sorun yaratabilir.

In [ ]:
def test_paraphrase_consistency(model, tokenizer, prompt_variants, choices, model_name=""):
    """Aynı sorunun farklı formülasyonlarına tutarlı cevap veriyor mu?"""
    winners = []
    details = []
    
    for prompt in prompt_variants:
        best_choice = None
        best_score = -float('inf')
        for choice in choices:
            lp = get_logprob(model, tokenizer, prompt, choice)
            if lp > best_score:
                best_score = lp
                best_choice = choice
        winners.append(best_choice)
        details.append({"Prompt": prompt[:50], "Winner": best_choice, "Score": round(best_score, 3)})
    
    unique = len(set(winners))
    consistent = unique == 1
    
    df = pd.DataFrame(details)
    df["Consistent"] = "PASS" if consistent else "FAIL"
    return df


paraphrase_tests = [
    {
        "variants": [
            "Merkez Bankası faiz oranını",
            "TCMB politika faizini",
            "Türkiye Cumhuriyet Merkez Bankası gösterge faizini",
        ],
        "choices": [" artırdı.", " düşürdü.", " sabit bıraktı."],
        "topic": "TCMB faiz"
    },
    {
        "variants": [
            "Enflasyon yükselince tüketicinin alım gücü",
            "Fiyatlar genel düzeyi artınca halkın satın alma gücü",
            "Yüksek enflasyon ortamında vatandaşın alım gücü",
        ],
        "choices": [" düşer.", " artar.", " değişmez."],
        "topic": "Enflasyon etkisi"
    },
]

In [ ]:
for name in MODELS:
    print(f"\n{'='*60}\n{name}\n{'='*60}")
    for test in paraphrase_tests:
        print(f"\n  Topic: {test['topic']}")
        display(test_paraphrase_consistency(
            MODELS[name], TOKENIZERS[name],
            test["variants"], test["choices"], name
        ))
clear_gpu()